<a href="https://colab.research.google.com/github/Ayseatci/DI725_Project/blob/dev/notebooks/04_phase3/DeiTTiny.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Land Cover Regression – DeiT Tiny Experiments

This notebook implements a multimodal regression framework that combines image features with textual descriptions to predict land cover distributions.

The experiments are designed to evaluate the contribution of textual information to visual models and the effect of different caption types listed in the following:
  - Text-only captions
  - Hybrid captions
  - Vision-generated captions

The impact of different lightweight text encoders, (MiniLM,DistilBERT, RemoteCLIP) are also observed.

The image backbone used in this notebook is Deit Tiny, which is fine-tuned during training. Text encoders are kept frozen, and features are combined using a gated fusion mechanism.

All experiments are run under a consistent setup (same split, seed, training configuration) to ensure fair comparison.

## Environment Setup

This section installs the required libraries for the experiments.
These packages provide the core functionality for building and training the multimodal model.

In [1]:
!pip install -q timm transformers wandb open_clip_torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00


## Data Loading and Setup

The dataset is loaded from Google Drive and extracted locally. This setup ensures that images and their corresponding textual descriptions can be accessed efficiently during training.

In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from torchvision import transforms

import wandb
import open_clip
from huggingface_hub import hf_hub_download
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
os.makedirs("/content/drive/MyDrive/2444347_DI725_term_project_phase3_predictions", exist_ok=True)

In [5]:
!unzip -q /content/drive/MyDrive/DI725_project_dataset_nomask.zip

In [6]:
LOCAL_DATA_ROOT = "/content/DI725_project_dataset_nomask"
CSV_PATH = os.path.join(LOCAL_DATA_ROOT, "captions.csv")
IMAGE_DIR = os.path.join(LOCAL_DATA_ROOT, "images")

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

print("CSV:", CSV_PATH)
print("Images:", IMAGE_DIR)
print("Number of images:", len(os.listdir(IMAGE_DIR)))

CSV: /content/DI725_project_dataset_nomask/captions.csv
Images: /content/DI725_project_dataset_nomask/images
Number of images: 10000


## Experiment Configuration

This cell defines the main experimental settings used throughout the notebook, including sample size, image size, batch size, number of epochs, learning rate, weight decay, early stopping patience, and random seed.

The target variables are the seven classes: Tree, Shrub, Grass, Crop, Built-up, Barren, and Water. The text_scale parameter controls the strength of the textual contribution during gated fusion.

In [7]:
CONFIG = {
    "sample_size": 10000,
    "img_size": 224,

    "batch_size": 32,
    "epochs": 15,
    "lr": 1e-4,
    "weight_decay": 1e-4,

    "model_name": "deit_tiny_patch16_224",

    "early_stopping_patience": 3,
    "grad_clip": 1.0,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,

    "text_scale": 0.7,

    # DataLoader
    "num_workers": 2,

    # Dataset
    "max_token_len": 128,

    # Model
    "regressor_hidden_dim": 256,
    "regressor_dropout": 0.25,

    # W&B
    "wandb_project": "DI725-Project_phase3_experiments",
    "wandb_project_scale": "DI725-Project_phase3_text_scale_exp",

    # Predictions output path
    "predictions_dir": "/content/drive/MyDrive/2444347_DI725_term_project_phase3_predictions",
}

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

## Reproducibility Setup

This cell fixes the random seed across Python, NumPy, and PyTorch to improve reproducibility. Using the same seed helps ensure that sampling, data splitting, and model training are comparable across experiments.

In [8]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])

## Data Loading and Text Cleaning

This section loads the caption and target data from the CSV file and prepares the text columns used in the multimodal experiments.

A cleaning function removes percentages, standalone numbers, and leakage related phrases from captions. This step prevents the model from directly using numerical target information contained in the text, while preserving semantic descriptions.

In [9]:
df = pd.read_csv(CSV_PATH)

TEXT_COLUMNS_RAW = [
    "hybrid_gemma3-4b",
    "text_qwen3-4b",
    "vision_gemma3-4b"
]

def clean_caption_no_numbers(text):
    text = str(text).lower()

    # remove percentages: 53%, 53 percent, 53 percentage
    text = re.sub(r"\b\d+(\.\d+)?\s*%|\b\d+(\.\d+)?\s*percent(age)?", "", text)

    # remove standalone numbers
    text = re.sub(r"\b\d+(\.\d+)?\b", "", text)

    leakage_words = [
        "covering", "coverage", "accounts for", "occupies",
        "making up", "comprises", "constitutes", "represents",
        "estimated", "approximately", "around", "about"
    ]

    for word in leakage_words:
        text = text.replace(word, "")

    text = re.sub(r"[%(),:;]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in TEXT_COLUMNS_RAW:
    clean_col = col.replace("-", "_") + "_clean"
    df[clean_col] = df[col].apply(clean_caption_no_numbers)

    print(clean_col)
    print(
        "Remaining numeric leakage:",
        df[clean_col].str.contains(
            r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
            regex=True,
            case=False
        ).sum()
    )

hybrid_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_504/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


text_qwen3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_504/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


vision_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_504/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


In [10]:
TEXT_COLS_CLEAN = [
    "hybrid_gemma3_4b_clean",
    "text_qwen3_4b_clean",
    "vision_gemma3_4b_clean"
]

## Stratified Sampling and Data Split

This section prepares the dataset for training by first filtering out rows with missing values in required columns (images, targets, and cleaned captions).

A dominant class is computed for each sample based on the maximum target value across land-cover classes. This is used to perform stratified sampling, ensuring that the class distribution is preserved.

## Train / Validation / Test Split

The sampled dataset is split into 80% training, 10% validation, 10% test

Stratification is applied based on the dominant class to maintain consistent class distribution across splits. This ensures fair and reliable evaluation of model performance.

Indices are reset after splitting to avoid indexing issues during data loading.

In [11]:
needed_cols = (
    ["filename"]
    + TARGET_COLS
    + ["hybrid_gemma3_4b_clean", "text_qwen3_4b_clean", "vision_gemma3_4b_clean"]
)

df = df.dropna(subset=needed_cols).copy()

df["dominant_class"] = df[TARGET_COLS].idxmax(axis=1)

# If available rows <= sample_size, use all rows
if len(df) <= CONFIG["sample_size"]:
    sample_df = df.sample(frac=1.0, random_state=CONFIG["seed"]).reset_index(drop=True)
else:
    sample_df, _ = train_test_split(
        df,
        train_size=CONFIG["sample_size"],
        stratify=df["dominant_class"],
        random_state=CONFIG["seed"]
    )
    sample_df = sample_df.reset_index(drop=True)

print("Sample size:", len(sample_df))

train_df, temp_df = train_test_split(
    sample_df,
    test_size=0.20,
    stratify=sample_df["dominant_class"],
    random_state=CONFIG["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dominant_class"],
    random_state=CONFIG["seed"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Sample size: 10000
Train: 8000
Val: 1000
Test: 1000


## Target Distribution Check

To verify that stratification is effective, this section computes summary statistics (mean and standard deviation) for each target class across the train, validation, and test splits.

This check ensures that the distributions of land cover classes remain consistent across splits, preventing bias in training or evaluation.

In [12]:
def check_target_distribution(train_df, val_df, test_df, target_cols):
    summary = []

    for name, data in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df)
    ]:
        row = {"split": name, "n": len(data)}

        for col in target_cols:
            row[f"{col}_mean"] = data[col].mean()
            row[f"{col}_std"] = data[col].std()

        summary.append(row)

    return pd.DataFrame(summary)

dist_summary = check_target_distribution(train_df, val_df, test_df, TARGET_COLS)
dist_summary

,split,n,Tree_mean,Tree_std,Shrub_mean,Shrub_std,Grass_mean,Grass_std,Crop_mean,Crop_std,Built-up_mean,Built-up_std,Barren_mean,Barren_std,Water_mean,Water_std
0,train,8000,28.8445,35.170320,0.828875,4.064469,45.367375,33.938682,17.95475,30.544088,1.139625,5.456130,4.192875,11.363640,1.672,9.244490
1,val,1000,28.8610,35.001154,0.818000,3.981297,45.064000,34.437681,18.15700,30.993789,1.190000,4.985053,4.181000,11.631231,1.729,9.926837
2,test,1000,28.1980,34.791741,0.941000,4.581103,45.318000,34.315075,18.14000,30.799334,1.095000,4.951615,4.508000,12.302898,1.800,9.461793


In [13]:
dominant_dist = pd.DataFrame({
    "train": train_df["dominant_class"].value_counts(normalize=True),
    "val": val_df["dominant_class"].value_counts(normalize=True),
    "test": test_df["dominant_class"].value_counts(normalize=True)
}).fillna(0)

dominant_dist

,train,val,test
dominant_class,,,
Grass,0.470375,0.470,0.470
Tree,0.303750,0.304,0.303
Crop,0.180250,0.181,0.180
Barren,0.019250,0.019,0.020
Water,0.017750,0.018,0.018
Built-up,0.005875,0.006,0.006
Shrub,0.002750,0.002,0.003


## Image Transformations

This section defines preprocessing pipelines for training and evaluation.

Training transformations include resizing, random horizontal and vertical flips for data augmentation, and normalization using ImageNet statistics.

Evaluation transformations apply resizing and normalization only, ensuring consistent input without augmentation during validation and testing.

In [ ]:
train_tfms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # mean and sdt values of ImageNet channel
                         std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## Multimodal Dataset

This dataset class prepares inputs for multimodal training. For each sample the image is loaded and transformed and target values are extracted as a multi-output regression vector.
If a text column is provided, captions are tokenized using a pretrained tokenizer

The dataset returns, image tensor, target vector, tokenized text inputs. This structure enables joint processing of image and text features in the model.

In [16]:
class LandCoverDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, tokenizer=None, text_col=None, max_len=CONFIG["max_token_len"]):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        item = {
            "image": image,
            "target": target
        }

        if self.tokenizer is not None and self.text_col is not None:
            text = str(row[self.text_col])

            enc = self.tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )

            item["input_ids"] = enc["input_ids"].squeeze(0)
            item["attention_mask"] = enc["attention_mask"].squeeze(0)

        return item

In [17]:
class LandCoverRawTextDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, text_col=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        text = str(row[self.text_col])

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "image": image,
            "text": text,
            "target": target
        }

## Image Only Baseline Model

This model serves as a baseline using only visual information.

A pretrained vision backbone (DeiT Tiny) is used to extract image features, which are then passed through a regression head to predict land-cover targets.

The regression head consists of layer normalization, fully connected layers,ReLU activation and dropout for regularization

This baseline allows comparison against multimodal models to measure the contribution of textual information. The backbone is initialized with pretrained weights and fine-tuned during training.

In [18]:
class ImageOnlyModel(nn.Module):
    def __init__(self, model_name, num_outputs=len(TARGET_COLS)):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.backbone.num_features

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def forward(self, image):
        image_features = self.backbone(image)
        return self.regressor(image_features)

## Multimodal Model (Image + Text Fusion)

This model extends the image-only baseline by incorporating textual information through a gated fusion mechanism.

The architecture consists of:
- A pretrained vision backbone (Deit Tiny) for image feature extraction
- A pretrained text encoder (MiniLM / DistilBERT/ RemoteCLIP) for caption representation
- A projection layer to map text features into the image feature space

The text encoder is kept frozen during training to reduce computational cost and ensure stable representations.

To combine modalities, a gating mechanism is used. Image and text features are concatenated and passed through a learnable gate. The gate controls how much textual information contributes to the final representation. Moreover, a scaling factor further adjusts the strength of text features.

The final fused representation is computed as:

    fused = image_features + scale × gate × text_features

This design allows the model to adaptively use textual information depending on its relevance.

In [19]:
class ImageTextGatedFrozenScaledModel(nn.Module):
    def __init__(
        self,
        image_model_name,
        text_model_name,
        num_outputs=len(TARGET_COLS),
        text_scale=CONFIG["text_scale"]
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_backbone = AutoModel.from_pretrained(text_model_name)

        for p in self.text_backbone.parameters():
            p.requires_grad = False

        text_dim = self.text_backbone.config.hidden_size

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

    def forward(self, image, input_ids, attention_mask):
        image_features = self.image_backbone(image)

        with torch.no_grad():
            text_output = self.text_backbone(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        text_features = self.mean_pooling(text_output, attention_mask)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused_features = image_features + self.text_scale * gate * text_features

        return self.regressor(fused_features)

In [20]:
class RemoteCLIPTextEncoder(nn.Module):
    def __init__(self, model_name="ViT-B-32"):
        super().__init__()

        self.model, _, _ = open_clip.create_model_and_transforms(
            model_name,
            pretrained=None
        )

        self.tokenizer = open_clip.get_tokenizer(model_name)

        ckpt_path = hf_hub_download(
            repo_id="chendelong/RemoteCLIP",
            filename=f"RemoteCLIP-{model_name}.pt"
        )

        ckpt = torch.load(ckpt_path, map_location="cpu")

        if "state_dict" in ckpt:
            ckpt = ckpt["state_dict"]

        cleaned = {}
        for k, v in ckpt.items():
            k = k.replace("module.", "").replace("model.", "")
            cleaned[k] = v

        self.model.load_state_dict(cleaned, strict=False)

        print("RemoteCLIP weights loaded")

        for p in self.model.parameters():
            p.requires_grad = False

        self.model.eval()

    def forward(self, texts):
        device = next(self.model.parameters()).device
        tokens = self.tokenizer(list(texts)).to(device)

        with torch.no_grad():
            features = self.model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)

        return features

In [21]:
class DeitTinyRemoteCLIPFusion(nn.Module):
    def __init__(
        self,
        image_model_name,
        num_outputs=len(TARGET_COLS),
        text_scale=CONFIG["text_scale"]
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_encoder = RemoteCLIPTextEncoder(model_name="ViT-B-32")

        with torch.no_grad():
            dummy_text = ["satellite image of land cover"]
            dummy_features = self.text_encoder(dummy_text)
            text_dim = dummy_features.shape[1]

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def forward(self, image, texts):
        image_features = self.image_backbone(image)

        text_features = self.text_encoder(texts)
        text_features = text_features.to(image_features.device)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused = image_features + self.text_scale * gate * text_features

        return self.regressor(fused)

## Evaluation Functions

This section defines evaluation procedures for both image only and multimodal models.

Mean Absolute Error (MAE) is used as the primary evaluation metric. Root Mean Squared Error (RMSE) is also computed for additional analysis. Furthermore, Class-wise MAE is calculated to analyze performance across land-cover categories

These metrics provide both overall and class-level performance insights.

In [22]:
@torch.no_grad()
def evaluate_model(model, loader, config, model_type):
    """
    Unified evaluation for all model types.
    model_type: "image" | "text" | "remoteclip"
    """
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images  = batch["image"].to(config["device"])
        targets = batch["target"].to(config["device"])

        if model_type == "image":
            preds = model(images)
        elif model_type == "text":
            input_ids      = batch["input_ids"].to(config["device"])
            attention_mask = batch["attention_mask"].to(config["device"])
            preds = model(images, input_ids, attention_mask)
        elif model_type == "remoteclip":
            preds = model(images, batch["text"])

        loss = criterion(preds, targets)
        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae       = mean_absolute_error(y_true, y_pred)
    rmse      = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae, y_pred, y_true

## Training: Image-Only Model

This function defines the training loop for the image-only baseline. The loss function is Mean Absolute Error (L1 loss). The AdamW optimizer with weight decay is used.

Training progress is logged using Weights & Biases (W&B).

## Training: Multimodal Model

This function extends the training process to multimodal inputs.

Differences from image only training is that the image and tokenized text inputs are used. Moreover, the multimodal forward pass combines features using gated fusion.

The same training configuration is maintained. This ensures a fair comparison between image only and multimodal models.

In [23]:
def train_model(model, train_loader, val_loader, config, model_type):
    """
    Unified training loop for all model types.
    model_type: "image" | "text" | "remoteclip"
    """
    criterion = nn.L1Loss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae    = float("inf")
    best_state      = None
    patience_counter = 0
    history         = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images  = batch["image"].to(config["device"])
            targets = batch["target"].to(config["device"])

            if model_type == "image":
                preds = model(images)
            elif model_type == "text":
                input_ids      = batch["input_ids"].to(config["device"])
                attention_mask = batch["attention_mask"].to(config["device"])
                preds = model(images, input_ids, attention_mask)
            elif model_type == "remoteclip":
                preds = model(images, batch["text"])

            loss = criterion(preds, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _, _, _ = evaluate_model(
            model, val_loader, config, model_type
        )

        history.append({
            "epoch":      epoch + 1,
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "val_mae":    val_mae,
            "val_rmse":   val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch":      epoch + 1,
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "val_mae":    val_mae,
            "val_rmse":   val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae     = val_mae
            best_state       = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_val_mae

## Experiment: Image Only Baseline

This function runs the image only experiment using the DeiTTiny backbone.

Metrics (MAE, RMSE, and class-wise MAE) are logged using Weights & Biases (W&B).

This experiment serves as the baseline for comparing multimodal configurations.

In [24]:
def run_deittiny_image_only(
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":   "deittiny_image_only",
        "fusion":       "none",
        "text_column":  "none",
        "text_model":   "none",
    }

    wandb.init(
        project=wandb_project,
        name="deittiny_image_only",
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms)
    val_ds   = LandCoverDataset(val_df,   IMAGE_DIR, transform=eval_tfms)
    test_ds  = LandCoverDataset(test_df,  IMAGE_DIR, transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = ImageOnlyModel(
        model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS)
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="image")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="image"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_image_only.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment": "deittiny_image_only",
        "text_model": "none",
        "test_mae":   test_mae,
        "test_rmse":  test_rmse,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

## Experiment: Multimodal (Image + Text)

This function runs multimodal experiments by combining image features with textual information.

Different caption sources and text encoders (MiniLM, DistilBERT) are utilized

For each configuration the model is initialized with a gated fusion mechanism, text contribution is controlled via a scaling factor and the same training and evaluation pipeline is applied

Results are logged to W&B, enabling comparison across different text sources and encoders. This setup allows systematic evaluation of how textual information impacts model performance.

In [25]:
def run_deittiny_transformer_text(
    text_col,
    text_model,
    text_scale=CONFIG["text_scale"],
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":   "deittiny_transformer_text",
        "fusion":       "gated_frozen_scaled",
        "text_column":  text_col,
        "text_model":   text_model,
        "text_scale":   text_scale,
    }

    run_name = f"deittiny_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}"

    wandb.init(project=wandb_project, name=run_name, config=run_config, reinit=True)

    tokenizer = AutoTokenizer.from_pretrained(text_model)

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms, tokenizer=tokenizer, text_col=text_col)
    val_ds   = LandCoverDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)
    test_ds  = LandCoverDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = ImageTextGatedFrozenScaledModel(
        image_model_name=CONFIG["model_name"],
        text_model_name=text_model,
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="text")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="text"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse, "best_val_mae": best_val_mae})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  text_model,
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

In [26]:
def run_deittiny_remoteclip_text(
    text_col,
    text_scale=CONFIG["text_scale"],
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":  "deittiny_remoteclip_text",
        "fusion":      "gated_frozen_scaled",
        "text_column": text_col,
        "text_model":  "RemoteCLIP ViT-B-32",
        "text_scale":  text_scale,
    }

    run_name = f"deittiny_remoteclip_{text_col}_scale_{text_scale}"

    wandb.init(project=wandb_project, name=run_name, config=run_config, reinit=True)

    train_ds = LandCoverRawTextDataset(train_df, IMAGE_DIR, transform=train_tfms, text_col=text_col)
    val_ds   = LandCoverRawTextDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  text_col=text_col)
    test_ds  = LandCoverRawTextDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = DeitTinyRemoteCLIPFusion(
        image_model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="remoteclip")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="remoteclip"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_{text_col}_remoteclip_scale_{text_scale}.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse, "best_val_mae": best_val_mae})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  "RemoteCLIP ViT-B-32",
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

# Running Experiments

In [27]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayseatci00 (ayseatci00-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [28]:
results = []

## Image Only Experiment

In [29]:
results.append(run_deittiny_image_only(save_predictions=True))
pd.DataFrame(results)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

Epoch 1/15 | Train Loss: 12.6245 | Val MAE: 11.0509 | Val RMSE: 24.9388
Epoch 2/15 | Train Loss: 9.8265 | Val MAE: 8.0198 | Val RMSE: 18.4197
Epoch 3/15 | Train Loss: 7.0133 | Val MAE: 5.6315 | Val RMSE: 13.4679
Epoch 4/15 | Train Loss: 5.3503 | Val MAE: 4.1578 | Val RMSE: 10.4887
Epoch 5/15 | Train Loss: 4.4420 | Val MAE: 3.6188 | Val RMSE: 9.3846
Epoch 6/15 | Train Loss: 3.9721 | Val MAE: 3.5339 | Val RMSE: 8.9386
Epoch 7/15 | Train Loss: 3.7488 | Val MAE: 3.2770 | Val RMSE: 8.5151
Epoch 8/15 | Train Loss: 3.5117 | Val MAE: 3.2309 | Val RMSE: 8.3319
Epoch 9/15 | Train Loss: 3.3785 | Val MAE: 3.0144 | Val RMSE: 7.9882
Epoch 10/15 | Train Loss: 3.2075 | Val MAE: 2.9407 | Val RMSE: 8.2967
Epoch 11/15 | Train Loss: 3.1060 | Val MAE: 2.8059 | Val RMSE: 7.4743
Epoch 12/15 | Train Loss: 2.9839 | Val MAE: 2.7008 | Val RMSE: 7.4873
Epoch 13/15 | Train Loss: 2.8625 | Val MAE: 2.9259 | Val RMSE: 8.1948
Epoch 14/15 | Train Loss: 2.7580 | Val MAE: 2.6718 | Val RMSE: 7.1749
Epoch 15/15 | Train Los

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▅▃▂▂▂▂▁▁▁▁▁▁▁▁
val_mae,█▅▃▂▂▂▂▁▁▁▁▁▁▁▁
val_rmse,█▅▃▂▂▂▂▁▁▁▁▁▁▁▁
epoch,15
test_loss,2.79605
test_mae,2.79605


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,deittiny_image_only,none,2.796047,7.349051,2.987114,0.969559,6.558145,4.593832,0.928737,2.583842,0.951103


## MiniLM Experiments

In [30]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_deittiny_transformer_text(
            text_col=text_col,
            text_model="sentence-transformers/all-MiniLM-L6-v2",
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 12.7346 | Val MAE: 11.3408 | Val RMSE: 25.3807
Epoch 2/15 | Train Loss: 10.0492 | Val MAE: 8.4016 | Val RMSE: 19.1864
Epoch 3/15 | Train Loss: 7.4063 | Val MAE: 5.8762 | Val RMSE: 13.9626
Epoch 4/15 | Train Loss: 5.6772 | Val MAE: 4.4926 | Val RMSE: 11.0045
Epoch 5/15 | Train Loss: 4.6069 | Val MAE: 3.5346 | Val RMSE: 9.0383
Epoch 6/15 | Train Loss: 4.0799 | Val MAE: 3.6680 | Val RMSE: 9.0382
Epoch 7/15 | Train Loss: 3.7562 | Val MAE: 3.1577 | Val RMSE: 8.7159
Epoch 8/15 | Train Loss: 3.5582 | Val MAE: 3.3312 | Val RMSE: 8.4484
Epoch 9/15 | Train Loss: 3.4234 | Val MAE: 3.2130 | Val RMSE: 8.5212
Epoch 10/15 | Train Loss: 3.2677 | Val MAE: 2.8814 | Val RMSE: 8.0066
Epoch 11/15 | Train Loss: 3.1178 | Val MAE: 3.0884 | Val RMSE: 8.3632
Epoch 12/15 | Train Loss: 3.0144 | Val MAE: 2.8995 | Val RMSE: 7.8329
Epoch 13/15 | Train Loss: 2.8633 | Val MAE: 2.8529 | Val RMSE: 7.8091
Epoch 14/15 | Train Loss: 2.7785 | Val MAE: 2.7421 | Val RMSE: 7.6916
Epoch 15/15 | Train Lo

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▆▄▂▂▂▁▂▁▁▁▁▁▁▁
val_mae,█▆▄▂▂▂▁▂▁▁▁▁▁▁▁
val_rmse,█▆▄▂▂▂▂▁▁▁▁▁▁▁▁
best_val_mae,2.668
epoch,15


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,sentence-transformers/all-MiniLM-L6-v2,2.446763,6.133510,3.423577,0.947782,5.103002,3.296342,0.866256,2.329579,1.160801,text_qwen3_4b_clean,0.7,2.365143,"[3.423577, 0.9477815, 5.1030016, 3.296342, 0.8..."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.481531,6.366873,3.019650,0.960672,5.353498,3.864892,0.811730,2.473203,0.887076,hybrid_gemma3_4b_clean,0.7,2.472036,"[3.01965, 0.96067154, 5.3534975, 3.864892, 0.8..."
0,deittiny_image_only,none,2.796047,7.349051,2.987114,0.969559,6.558145,4.593832,0.928737,2.583842,0.951103,NaN,NaN,NaN,NaN
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.816919,7.619487,2.986358,0.946177,6.259780,4.905833,0.772024,2.564888,1.283373,vision_gemma3_4b_clean,0.7,2.668003,"[2.9863584, 0.9461766, 6.2597804, 4.9058332, 0..."


## Remote CLIP Experiments

In [31]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_deittiny_remoteclip_text(
            text_col=text_col,
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

RemoteCLIP-ViT-B-32.pt:   0%|          | 0.00/605M [00:00<?, ?B/s]

RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 12.7813 | Val MAE: 11.3083 | Val RMSE: 25.4772
Epoch 2/15 | Train Loss: 10.1799 | Val MAE: 8.5116 | Val RMSE: 19.3695
Epoch 3/15 | Train Loss: 7.4729 | Val MAE: 5.9025 | Val RMSE: 14.0730
Epoch 4/15 | Train Loss: 5.6398 | Val MAE: 4.5018 | Val RMSE: 10.9580
Epoch 5/15 | Train Loss: 4.6434 | Val MAE: 3.6460 | Val RMSE: 9.2444
Epoch 6/15 | Train Loss: 4.0570 | Val MAE: 3.1535 | Val RMSE: 8.3993
Epoch 7/15 | Train Loss: 3.7118 | Val MAE: 3.0617 | Val RMSE: 8.2601
Epoch 8/15 | Train Loss: 3.5539 | Val MAE: 3.0154 | Val RMSE: 8.1810
Epoch 9/15 | Train Loss: 3.3778 | Val MAE: 2.8732 | Val RMSE: 7.8311
Epoch 10/15 | Train Loss: 3.2167 | Val MAE: 2.8821 | Val RMSE: 7.6504
Epoch 11/15 | Train Loss: 3.0520 | Val MAE: 2.6203 | Val RMSE: 7.3770
Epoch 12/15 | Train Loss: 2.9525 | Val MAE: 2.6620 | Val RMSE: 7.2733
Epoch 13/15 | Train Loss: 2.8488 | Val MAE: 2.6591 | Val RMSE: 7.1317
Epoch 14/15 | Train Loss: 2.6616 | Val MAE: 2.5151 | Val RMSE: 6.9

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▁▁▁▁▁▁▁▁▁
val_mae,█▆▄▃▂▂▁▁▁▁▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
best_val_mae,2.51513
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 12.7605 | Val MAE: 11.3045 | Val RMSE: 25.3289
Epoch 2/15 | Train Loss: 10.0891 | Val MAE: 8.4202 | Val RMSE: 19.2370
Epoch 3/15 | Train Loss: 7.3937 | Val MAE: 5.8263 | Val RMSE: 14.0575
Epoch 4/15 | Train Loss: 5.5403 | Val MAE: 4.4177 | Val RMSE: 11.0042
Epoch 5/15 | Train Loss: 4.5702 | Val MAE: 3.7090 | Val RMSE: 9.3589
Epoch 6/15 | Train Loss: 4.0160 | Val MAE: 3.2449 | Val RMSE: 8.5340
Epoch 7/15 | Train Loss: 3.6567 | Val MAE: 2.9890 | Val RMSE: 8.0013
Epoch 8/15 | Train Loss: 3.4167 | Val MAE: 2.7424 | Val RMSE: 7.6212
Epoch 9/15 | Train Loss: 3.2671 | Val MAE: 2.7252 | Val RMSE: 7.4358
Epoch 10/15 | Train Loss: 3.0787 | Val MAE: 2.6879 | Val RMSE: 7.1496
Epoch 11/15 | Train Loss: 2.9327 | Val MAE: 2.5935 | Val RMSE: 7.0342
Epoch 12/15 | Train Loss: 2.7942 | Val MAE: 2.7608 | Val RMSE: 7.3695
Epoch 13/15 | Train Loss: 2.6957 | Val MAE: 2.5640 | Val RMSE: 6.8647
Epoch 14/15 | Train Loss: 2.5757 | Val MAE: 2.5982 | Val RMSE: 6.5

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▁▁▁▁▁▁▁▁▁
val_mae,█▆▄▃▂▂▁▁▁▁▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
best_val_mae,2.44374
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 12.7636 | Val MAE: 11.2906 | Val RMSE: 25.4237
Epoch 2/15 | Train Loss: 10.0821 | Val MAE: 8.3609 | Val RMSE: 19.1917
Epoch 3/15 | Train Loss: 7.5281 | Val MAE: 5.9014 | Val RMSE: 14.1064
Epoch 4/15 | Train Loss: 5.7185 | Val MAE: 4.5510 | Val RMSE: 11.0303
Epoch 5/15 | Train Loss: 4.6727 | Val MAE: 3.7340 | Val RMSE: 9.3206
Epoch 6/15 | Train Loss: 4.1610 | Val MAE: 3.4461 | Val RMSE: 8.7562
Epoch 7/15 | Train Loss: 3.8169 | Val MAE: 3.1098 | Val RMSE: 8.4506
Epoch 8/15 | Train Loss: 3.5712 | Val MAE: 3.2035 | Val RMSE: 8.1855
Epoch 9/15 | Train Loss: 3.4222 | Val MAE: 2.9435 | Val RMSE: 8.0446
Epoch 10/15 | Train Loss: 3.2999 | Val MAE: 2.8468 | Val RMSE: 7.7661
Epoch 11/15 | Train Loss: 3.1496 | Val MAE: 2.8433 | Val RMSE: 7.7978
Epoch 12/15 | Train Loss: 3.0306 | Val MAE: 2.8291 | Val RMSE: 7.7047
Epoch 13/15 | Train Loss: 2.9204 | Val MAE: 2.7085 | Val RMSE: 7.4888
Epoch 14/15 | Train Loss: 2.7878 | Val MAE: 2.8237 | Val RMSE: 7.7

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▁▁▁▁▁▁▁▁▁
val_mae,█▆▄▃▂▂▁▁▁▁▁▁▁▁▁
val_rmse,█▆▄▂▂▂▁▁▁▁▁▁▁▁▁
best_val_mae,2.70851
epoch,15


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,sentence-transformers/all-MiniLM-L6-v2,2.446763,6.133510,3.423577,0.947782,5.103002,3.296342,0.866256,2.329579,1.160801,text_qwen3_4b_clean,0.7,2.365143,"[3.423577, 0.9477815, 5.1030016, 3.296342, 0.8..."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.481531,6.366873,3.019650,0.960672,5.353498,3.864892,0.811730,2.473203,0.887076,hybrid_gemma3_4b_clean,0.7,2.472036,"[3.01965, 0.96067154, 5.3534975, 3.864892, 0.8..."
5,deittiny_remoteclip_text_qwen3_4b_clean_scale_0.7,RemoteCLIP ViT-B-32,2.494713,6.568071,2.736492,0.958111,5.639515,3.862867,1.025586,2.480574,0.759841,text_qwen3_4b_clean,0.7,2.443738,"[2.7364922, 0.9581109, 5.639515, 3.8628674, 1...."
4,deittiny_remoteclip_hybrid_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.552218,6.626449,3.399971,0.951334,5.608590,3.704952,0.897112,2.438674,0.864892,hybrid_gemma3_4b_clean,0.7,2.515125,"[3.3999712, 0.9513345, 5.6085896, 3.7049525, 0..."
0,deittiny_image_only,none,2.796047,7.349051,2.987114,0.969559,6.558145,4.593832,0.928737,2.583842,0.951103,NaN,NaN,NaN,NaN
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.816919,7.619487,2.986358,0.946177,6.259780,4.905833,0.772024,2.564888,1.283373,vision_gemma3_4b_clean,0.7,2.668003,"[2.9863584, 0.9461766, 6.2597804, 4.9058332, 0..."
6,deittiny_remoteclip_vision_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.878144,7.772720,3.072003,0.950671,6.550913,4.927624,0.969234,2.691180,0.985381,vision_gemma3_4b_clean,0.7,2.708505,"[3.0720034, 0.9506711, 6.550913, 4.927624, 0.9..."


## DistilBERT Experiments

In [32]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_deittiny_transformer_text(
            text_col=text_col,
            text_model="distilbert-base-uncased",
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 12.8825 | Val MAE: 11.4317 | Val RMSE: 25.7609
Epoch 2/15 | Train Loss: 10.2716 | Val MAE: 8.7220 | Val RMSE: 19.7169
Epoch 3/15 | Train Loss: 7.6762 | Val MAE: 6.4486 | Val RMSE: 14.5303
Epoch 4/15 | Train Loss: 5.8120 | Val MAE: 4.7021 | Val RMSE: 10.8942
Epoch 5/15 | Train Loss: 4.6681 | Val MAE: 3.9514 | Val RMSE: 9.6811
Epoch 6/15 | Train Loss: 4.0832 | Val MAE: 3.2958 | Val RMSE: 8.5700
Epoch 7/15 | Train Loss: 3.8538 | Val MAE: 3.2115 | Val RMSE: 8.5448
Epoch 8/15 | Train Loss: 3.6532 | Val MAE: 3.2739 | Val RMSE: 8.4809
Epoch 9/15 | Train Loss: 3.4624 | Val MAE: 3.0188 | Val RMSE: 8.0623
Epoch 10/15 | Train Loss: 3.3098 | Val MAE: 2.9699 | Val RMSE: 8.1046
Epoch 11/15 | Train Loss: 3.1535 | Val MAE: 3.2554 | Val RMSE: 8.6400
Epoch 12/15 | Train Loss: 2.9863 | Val MAE: 2.8523 | Val RMSE: 7.6418
Epoch 13/15 | Train Loss: 2.8730 | Val MAE: 2.9172 | Val RMSE: 7.7980
Epoch 14/15 | Train Loss: 2.7793 | Val MAE: 2.6846 | Val RMSE: 7.5286
Epoch 15/15 | Train Lo

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▆▄▃▂▁▁▁▁▁▁▁▁▁▁
val_mae,█▆▄▃▂▁▁▁▁▁▁▁▁▁▁
val_rmse,█▆▄▂▂▁▁▁▁▁▁▁▁▁▁
best_val_mae,2.68464
epoch,15


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
8,deittiny_text_qwen3_4b_clean_distilbert-base-u...,distilbert-base-uncased,2.293479,5.903532,2.662381,0.957869,4.988350,3.265521,0.731270,2.582826,0.866134,text_qwen3_4b_clean,0.7,2.256033,"[2.662381, 0.95786935, 4.98835, 3.2655208, 0.7..."
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,sentence-transformers/all-MiniLM-L6-v2,2.446763,6.133510,3.423577,0.947782,5.103002,3.296342,0.866256,2.329579,1.160801,text_qwen3_4b_clean,0.7,2.365143,"[3.423577, 0.9477815, 5.1030016, 3.296342, 0.8..."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.481531,6.366873,3.019650,0.960672,5.353498,3.864892,0.811730,2.473203,0.887076,hybrid_gemma3_4b_clean,0.7,2.472036,"[3.01965, 0.96067154, 5.3534975, 3.864892, 0.8..."
5,deittiny_remoteclip_text_qwen3_4b_clean_scale_0.7,RemoteCLIP ViT-B-32,2.494713,6.568071,2.736492,0.958111,5.639515,3.862867,1.025586,2.480574,0.759841,text_qwen3_4b_clean,0.7,2.443738,"[2.7364922, 0.9581109, 5.639515, 3.8628674, 1...."
4,deittiny_remoteclip_hybrid_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.552218,6.626449,3.399971,0.951334,5.608590,3.704952,0.897112,2.438674,0.864892,hybrid_gemma3_4b_clean,0.7,2.515125,"[3.3999712, 0.9513345, 5.6085896, 3.7049525, 0..."
7,deittiny_hybrid_gemma3_4b_clean_distilbert-bas...,distilbert-base-uncased,2.629013,6.875816,2.560386,0.955085,6.321153,4.539212,0.808565,2.518461,0.700228,hybrid_gemma3_4b_clean,0.7,2.567945,"[2.5603864, 0.955085, 6.3211527, 4.5392118, 0...."
0,deittiny_image_only,none,2.796047,7.349051,2.987114,0.969559,6.558145,4.593832,0.928737,2.583842,0.951103,NaN,NaN,NaN,NaN
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.816919,7.619487,2.986358,0.946177,6.259780,4.905833,0.772024,2.564888,1.283373,vision_gemma3_4b_clean,0.7,2.668003,"[2.9863584, 0.9461766, 6.2597804, 4.9058332, 0..."
9,deittiny_vision_gemma3_4b_clean_distilbert-bas...,distilbert-base-uncased,2.856610,7.785571,2.949629,0.962799,6.585811,5.276410,0.781482,2.677168,0.762968,vision_gemma3_4b_clean,0.7,2.684645,"[2.9496293, 0.96279925, 6.585811, 5.27641, 0.7..."
6,deittiny_remoteclip_vision_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.878144,7.772720,3.072003,0.950671,6.550913,4.927624,0.969234,2.691180,0.985381,vision_gemma3_4b_clean,0.7,2.708505,"[3.0720034, 0.9506711, 6.550913, 4.927624, 0.9..."


## Deit Tiny Experiment Results Summary

In [34]:
results_df = pd.DataFrame(results).sort_values("test_mae")
results_df

,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
8,deittiny_text_qwen3_4b_clean_distilbert-base-u...,distilbert-base-uncased,2.293479,5.903532,2.662381,0.957869,4.988350,3.265521,0.731270,2.582826,0.866134,text_qwen3_4b_clean,0.7,2.256033,"[2.662381, 0.95786935, 4.98835, 3.2655208, 0.7..."
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,sentence-transformers/all-MiniLM-L6-v2,2.446763,6.133510,3.423577,0.947782,5.103002,3.296342,0.866256,2.329579,1.160801,text_qwen3_4b_clean,0.7,2.365143,"[3.423577, 0.9477815, 5.1030016, 3.296342, 0.8..."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.481531,6.366873,3.019650,0.960672,5.353498,3.864892,0.811730,2.473203,0.887076,hybrid_gemma3_4b_clean,0.7,2.472036,"[3.01965, 0.96067154, 5.3534975, 3.864892, 0.8..."
5,deittiny_remoteclip_text_qwen3_4b_clean_scale_0.7,RemoteCLIP ViT-B-32,2.494713,6.568071,2.736492,0.958111,5.639515,3.862867,1.025586,2.480574,0.759841,text_qwen3_4b_clean,0.7,2.443738,"[2.7364922, 0.9581109, 5.639515, 3.8628674, 1...."
4,deittiny_remoteclip_hybrid_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.552218,6.626449,3.399971,0.951334,5.608590,3.704952,0.897112,2.438674,0.864892,hybrid_gemma3_4b_clean,0.7,2.515125,"[3.3999712, 0.9513345, 5.6085896, 3.7049525, 0..."
7,deittiny_hybrid_gemma3_4b_clean_distilbert-bas...,distilbert-base-uncased,2.629013,6.875816,2.560386,0.955085,6.321153,4.539212,0.808565,2.518461,0.700228,hybrid_gemma3_4b_clean,0.7,2.567945,"[2.5603864, 0.955085, 6.3211527, 4.5392118, 0...."
0,deittiny_image_only,none,2.796047,7.349051,2.987114,0.969559,6.558145,4.593832,0.928737,2.583842,0.951103,NaN,NaN,NaN,NaN
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.816919,7.619487,2.986358,0.946177,6.259780,4.905833,0.772024,2.564888,1.283373,vision_gemma3_4b_clean,0.7,2.668003,"[2.9863584, 0.9461766, 6.2597804, 4.9058332, 0..."
9,deittiny_vision_gemma3_4b_clean_distilbert-bas...,distilbert-base-uncased,2.856610,7.785571,2.949629,0.962799,6.585811,5.276410,0.781482,2.677168,0.762968,vision_gemma3_4b_clean,0.7,2.684645,"[2.9496293, 0.96279925, 6.585811, 5.27641, 0.7..."
6,deittiny_remoteclip_vision_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.878144,7.772720,3.072003,0.950671,6.550913,4.927624,0.969234,2.691180,0.985381,vision_gemma3_4b_clean,0.7,2.708505,"[3.0720034, 0.9506711, 6.550913, 4.927624, 0.9..."


Experiments are conducted using three caption types (text only, hybrid, and vision based) and three text encoders (MiniLM, RemoteCLIP and DistilBERT). A fixed text scaling factor is applied to control the contribution of textual features in the fusion process.

## Text Scale Experiments

The best caption × encoder combination for DeiT-Tiny based on
validation MAE at scale 0.7 is selected.

In [35]:
# Filter out image-only row — selection is over multimodal configs only
multimodal_results = [r for r in results if r.get("text_model") not in [None, "none"]]
results_df_multimodal = pd.DataFrame(multimodal_results)

best_config = results_df_multimodal.loc[results_df_multimodal["val_mae"].idxmin()]

print("Best configuration for DeiT-Tiny (by validation MAE)")
print(f"  Text column : {best_config['text_column']}")
print(f"  Text model  : {best_config['text_model']}")
print(f"  Val MAE     : {best_config['val_mae']:.4f}")
print(f"  Test MAE    : {best_config['test_mae']:.4f}")

BEST_TEXT_COL   = best_config["text_column"]
BEST_TEXT_MODEL = best_config["text_model"]
IS_REMOTECLIP   = (BEST_TEXT_MODEL == "RemoteCLIP ViT-B-32")

Best configuration for DeiT-Tiny (by validation MAE)
  Text column : text_qwen3_4b_clean
  Text model  : distilbert-base-uncased
  Val MAE     : 2.2560
  Test MAE    : 2.2935


## Text Scale Ablation

Using the best configuration selected above, we sweep text_scale across
[0.3, 0.5, 0.7, 1.0, 1.5].

In [36]:
TEXT_SCALES = [0.3, 0.5, 0.7, 1.0, 1.5]

scale_results = []

for scale in TEXT_SCALES:
    print(f"\nRunning scale={scale} | col={BEST_TEXT_COL} | model={BEST_TEXT_MODEL}")

    if IS_REMOTECLIP:
        result = run_deittiny_remoteclip_text(
            text_col=BEST_TEXT_COL,
            text_scale=scale,
            wandb_project=CONFIG["wandb_project_scale"],
            save_predictions=True
        )
    else:
        result = run_deittiny_transformer_text(
            text_col=BEST_TEXT_COL,
            text_model=BEST_TEXT_MODEL,
            text_scale=scale,
            wandb_project=CONFIG["wandb_project_scale"],
            save_predictions=True
        )

    scale_results.append(result)

scale_df = pd.DataFrame(scale_results).sort_values("text_scale").reset_index(drop=True)
scale_df


Running scale=0.3 | col=text_qwen3_4b_clean | model=distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 12.8795 | Val MAE: 11.4407 | Val RMSE: 25.7188
Epoch 2/15 | Train Loss: 10.2895 | Val MAE: 8.6235 | Val RMSE: 19.3060
Epoch 3/15 | Train Loss: 7.5211 | Val MAE: 6.3321 | Val RMSE: 14.4419
Epoch 4/15 | Train Loss: 5.7296 | Val MAE: 4.6371 | Val RMSE: 10.8425
Epoch 5/15 | Train Loss: 4.5578 | Val MAE: 3.4931 | Val RMSE: 8.9030
Epoch 6/15 | Train Loss: 3.9985 | Val MAE: 3.5021 | Val RMSE: 8.9206
Epoch 7/15 | Train Loss: 3.7019 | Val MAE: 3.1676 | Val RMSE: 8.5136
Epoch 8/15 | Train Loss: 3.4767 | Val MAE: 2.9445 | Val RMSE: 7.9368
Epoch 9/15 | Train Loss: 3.2851 | Val MAE: 2.9388 | Val RMSE: 7.8657
Epoch 10/15 | Train Loss: 3.1520 | Val MAE: 2.7104 | Val RMSE: 7.3043
Epoch 11/15 | Train Loss: 2.9787 | Val MAE: 2.6844 | Val RMSE: 7.2226
Epoch 12/15 | Train Loss: 2.8326 | Val MAE: 2.4773 | Val RMSE: 6.7149
Epoch 13/15 | Train Loss: 2.7258 | Val MAE: 2.4587 | Val RMSE: 6.5861
Epoch 14/15 | Train Loss: 2.6119 | Val MAE: 2.4046 | Val RMSE: 6.4385
Epoch 15/15 | Train Lo

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_mae,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
best_val_mae,2.27801
epoch,15


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 12.8696 | Val MAE: 11.4109 | Val RMSE: 25.7052
Epoch 2/15 | Train Loss: 10.1785 | Val MAE: 8.6086 | Val RMSE: 19.3151
Epoch 3/15 | Train Loss: 7.3975 | Val MAE: 5.9429 | Val RMSE: 13.7357
Epoch 4/15 | Train Loss: 5.4509 | Val MAE: 4.0940 | Val RMSE: 9.8503
Epoch 5/15 | Train Loss: 4.3284 | Val MAE: 3.2894 | Val RMSE: 8.3757
Epoch 6/15 | Train Loss: 3.7565 | Val MAE: 3.1571 | Val RMSE: 8.1951
Epoch 7/15 | Train Loss: 3.5423 | Val MAE: 2.9637 | Val RMSE: 7.8140
Epoch 8/15 | Train Loss: 3.3786 | Val MAE: 3.1218 | Val RMSE: 7.8833
Epoch 9/15 | Train Loss: 3.1661 | Val MAE: 2.8581 | Val RMSE: 7.4318
Epoch 10/15 | Train Loss: 3.0475 | Val MAE: 2.6351 | Val RMSE: 7.1721
Epoch 11/15 | Train Loss: 2.8796 | Val MAE: 2.5605 | Val RMSE: 6.7425
Epoch 12/15 | Train Loss: 2.7555 | Val MAE: 2.4749 | Val RMSE: 6.5214
Epoch 13/15 | Train Loss: 2.6016 | Val MAE: 2.3050 | Val RMSE: 6.1207
Epoch 14/15 | Train Loss: 2.5231 | Val MAE: 2.2120 | Val RMSE: 5.8897
Epoch 15/15 | Train Los

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▆▄▂▂▂▂▂▂▁▁▁▁▁▁
val_mae,█▆▄▂▂▂▂▂▂▁▁▁▁▁▁
val_rmse,█▆▄▂▂▂▂▂▂▂▁▁▁▁▁
best_val_mae,2.19528
epoch,15


,experiment,text_column,text_model,text_scale,val_mae,test_mae,test_rmse,class_mae,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,0.3,2.278011,2.336504,6.103200,"[2.671483, 0.9526515, 5.1752744, 3.5377653, 0....",2.671483,0.952652,5.175274,3.537765,0.703265,2.555834,0.759257
1,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,0.5,2.310439,2.333532,6.087516,"[2.6551628, 0.95062625, 5.010062, 3.3819795, 0...",2.655163,0.950626,5.010062,3.381979,0.749305,2.620655,0.966936
2,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,0.7,2.235355,2.241823,5.827783,"[2.6065967, 0.95025945, 4.807175, 3.280339, 0....",2.606597,0.950259,4.807175,3.280339,0.748297,2.479741,0.820357
3,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,1.0,2.267722,2.293900,5.865313,"[2.6656463, 0.95553875, 4.937132, 3.464, 0.773...",2.665646,0.955539,4.937132,3.464000,0.773937,2.425903,0.835142
4,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,1.5,2.195277,2.252425,5.684266,"[2.7869682, 0.9509605, 4.9664755, 3.1569924, 0...",2.786968,0.950961,4.966475,3.156992,0.785930,2.426110,0.693535


## Text Scale Ablation — Overall MAE Summary

In [37]:
image_only_row = next(r for r in results if r.get("text_model") in [None, "none"])
baseline_mae   = image_only_row["test_mae"]

scale_df["delta_vs_baseline"] = (baseline_mae - scale_df["test_mae"]).round(4)
scale_df["pct_improvement"]   = ((scale_df["delta_vs_baseline"] / baseline_mae) * 100).round(2)

print(f"Image-only baseline test MAE : {baseline_mae:.4f}\n")
print(f"Best config : {BEST_TEXT_COL}  +  {BEST_TEXT_MODEL}\n")

display_cols = ["text_scale", "val_mae", "test_mae", "test_rmse", "delta_vs_baseline", "pct_improvement"]
scale_df[display_cols]

Image-only baseline test MAE : 2.7960

Best config : text_qwen3_4b_clean  +  distilbert-base-uncased



,text_scale,val_mae,test_mae,test_rmse,delta_vs_baseline,pct_improvement
0,0.3,2.278011,2.336504,6.103200,0.4595,16.43
1,0.5,2.310439,2.333532,6.087516,0.4625,16.54
2,0.7,2.235355,2.241823,5.827783,0.5542,19.82
3,1.0,2.267722,2.293900,5.865313,0.5021,17.96
4,1.5,2.195277,2.252425,5.684266,0.5436,19.44


## Text Scale Ablation — Class-wise MAE

In [38]:
class_mae_cols = [f"mae_{cls}" for cls in TARGET_COLS]
class_scale_df = scale_df[["text_scale"] + class_mae_cols].copy()
class_scale_df.columns = ["text_scale"] + TARGET_COLS

baseline_class_mae = np.array([image_only_row[f"mae_{cls}"] for cls in TARGET_COLS])

delta_rows = []
for _, row in class_scale_df.iterrows():
    row_mae = np.array([row[cls] for cls in TARGET_COLS])
    delta_rows.append(
        {"text_scale": row["text_scale"],
         **{cls: round(baseline_class_mae[i] - row_mae[i], 4) for i, cls in enumerate(TARGET_COLS)}}
    )

delta_class_df = pd.DataFrame(delta_rows)

print("Class-wise MAE per text scale")
print(class_scale_df.to_string(index=False))

print("\nClass-wise improvement vs image-only baseline (positive = better)")
print(delta_class_df.to_string(index=False))

Class-wise MAE per text scale
 text_scale     Tree    Shrub    Grass     Crop  Built-up   Barren    Water
        0.3 2.671483 0.952652 5.175274 3.537765  0.703265 2.555834 0.759257
        0.5 2.655163 0.950626 5.010062 3.381979  0.749305 2.620655 0.966936
        0.7 2.606597 0.950259 4.807175 3.280339  0.748297 2.479741 0.820357
        1.0 2.665646 0.955539 4.937132 3.464000  0.773937 2.425903 0.835142
        1.5 2.786968 0.950961 4.966475 3.156992  0.785930 2.426110 0.693535

Class-wise improvement vs image-only baseline (positive = better)
 text_scale   Tree  Shrub  Grass   Crop  Built-up  Barren   Water
        0.3 0.3156 0.0169 1.3829 1.0561    0.2255  0.0280  0.1918
        0.5 0.3320 0.0189 1.5481 1.2119    0.1794 -0.0368 -0.0158
        0.7 0.3805 0.0193 1.7510 1.3135    0.1804  0.1041  0.1307
        1.0 0.3215 0.0140 1.6210 1.1298    0.1548  0.1579  0.1160
        1.5 0.2001 0.0186 1.5917 1.4368    0.1428  0.1577  0.2576


## Full Results Summary — All DeiT-Tiny Experiments

In [39]:
scale_result_experiments = {r["experiment"] for r in scale_results}

all_results_df = pd.DataFrame(results + scale_results).copy()

all_results_df["experiment_type"] = all_results_df.apply(
    lambda r: "image_only"  if pd.isna(r.get("text_model")) or r.get("text_model") in [None, "none"]
    else ("scale_sweep"     if r.get("experiment") in scale_result_experiments
    else  "multimodal_0.7"),
    axis=1
)

display_cols = [
    "experiment_type", "text_column", "text_model",
    "text_scale", "val_mae", "test_mae", "test_rmse"
]

all_results_df[display_cols].sort_values(
    ["experiment_type", "test_mae"]
).reset_index(drop=True)

,experiment_type,text_column,text_model,text_scale,val_mae,test_mae,test_rmse
0,image_only,NaN,none,NaN,NaN,2.796047,7.349051
1,multimodal_0.7,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.365143,2.446763,6.133510
2,multimodal_0.7,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.472036,2.481531,6.366873
3,multimodal_0.7,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.443738,2.494713,6.568071
4,multimodal_0.7,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.515125,2.552218,6.626449
5,multimodal_0.7,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.567945,2.629013,6.875816
6,multimodal_0.7,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.668003,2.816919,7.619487
7,multimodal_0.7,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.684645,2.856610,7.785571
8,multimodal_0.7,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.708505,2.878144,7.772720
9,scale_sweep,text_qwen3_4b_clean,distilbert-base-uncased,0.7,2.235355,2.241823,5.827783


In [40]:
# Best configuration overall — across all experiments including scale sweep
all_multimodal = all_results_df[all_results_df["experiment_type"] != "image_only"]
best_overall = all_multimodal.loc[all_multimodal["val_mae"].idxmin()]

print("Best overall configuration (by validation MAE)")
print(f"  Experiment type : {best_overall['experiment_type']}")
print(f"  Text column     : {best_overall['text_column']}")
print(f"  Text model      : {best_overall['text_model']}")
print(f"  Text scale      : {best_overall['text_scale']}")
print(f"  Val MAE         : {best_overall['val_mae']:.4f}")
print(f"  Test MAE        : {best_overall['test_mae']:.4f}")
print(f"  Test RMSE       : {best_overall['test_rmse']:.4f}")

Best overall configuration (by validation MAE)
  Experiment type : scale_sweep
  Text column     : text_qwen3_4b_clean
  Text model      : distilbert-base-uncased
  Text scale      : 1.5
  Val MAE         : 2.1953
  Test MAE        : 2.2524
  Test RMSE       : 5.6843


In [41]:
BACKBONE_NAME = CONFIG["model_name"].replace("_patch16_224", "").replace("_", "")

save_path = os.path.join(CONFIG["predictions_dir"], f"all_results_{BACKBONE_NAME}.csv")
all_results_df.to_csv(save_path, index=False)
print("Saved results to Drive")

Saved results to Drive
